# Analisis Exploratorio y Perfilamiento de Datos - Data Warehouse EPN

**Asignatura**: Business Intelligence  
**Institucion**: Escuela Politecnica Nacional (EPN)  
**Objetivo**: Validar la consistencia, integridad referencial y distribucion dimensional de los datos institucionales extraidos del portal oficial.

In [ ]:
import sqlite3
from pathlib import Path
import pandas as pd
import matplotlib.pyplot as plt

# Configuracion de rutas
DB_PATH = Path('../data/epn_dwh.db')
PROCESSED_DIR = Path('../data/processed')

conn = sqlite3.connect(DB_PATH)
print('Conexion exitosa a base de datos relacional EPN DWH.')

## 1. Perfilamiento Cuantitativo de Tablas Dimensionales y de Hechos

In [ ]:
tables_query = "SELECT name FROM sqlite_master WHERE type='table' AND name NOT LIKE 'sqlite_%';"
tables = [t[0] for t in conn.execute(tables_query).fetchall()]

profiling_rows = []
for tbl in tables:
    df = pd.read_sql_query(f"SELECT * FROM {tbl}", conn)
    profiling_rows.append({
        'Tabla': tbl,
        'Registros': len(df),
        'Columnas': len(df.columns),
        'Nulos': df.isnull().sum().sum()
    })

df_profile = pd.DataFrame(profiling_rows)
display(df_profile)

## 2. Oferta Academica por Facultad y Nivel de Formacion

In [ ]:
query_oferta = """
SELECT 
    f.sigla AS Facultad,
    n.nombre_nivel AS Nivel,
    COUNT(fo.oferta_key) AS Cantidad_Programas
FROM fact_oferta_academica fo
JOIN dim_facultad f ON fo.facultad_key = f.facultad_key
JOIN dim_nivel_formacion n ON fo.nivel_key = n.nivel_key
GROUP BY f.sigla, n.nombre_nivel
ORDER BY Cantidad_Programas DESC;
"""
df_oferta = pd.read_sql_query(query_oferta, conn)
display(df_oferta.head(10))

# Grafico de barras apiladas por nivel
pivot_oferta = df_oferta.pivot(index='Facultad', columns='Nivel', values='Cantidad_Programas').fillna(0)
pivot_oferta.plot(kind='bar', figsize=(10, 5), title='Programas Ofertados por Facultad y Nivel')
plt.ylabel('Cantidad de Programas')
plt.xticks(rotation=45)
plt.grid(axis='y', linestyle='--', alpha=0.7)
plt.tight_layout()
plt.show()

## 3. Distribucion de Modalidades Pedagogicas

In [ ]:
query_modalidad = """
SELECT 
    m.nombre_modalidad AS Modalidad,
    COUNT(*) AS Total
FROM fact_oferta_academica fo
JOIN dim_modalidad m ON fo.modalidad_key = m.modalidad_key
GROUP BY m.nombre_modalidad;
"""
df_mod = pd.read_sql_query(query_modalidad, conn)
plt.figure(figsize=(6, 6))
plt.pie(df_mod['Total'], labels=df_mod['Modalidad'], autopct='%1.1f%%', startangle=140)
plt.title('Distribucion de Modalidades de Formacion en la EPN')
plt.show()

## 4. Distribucion de Autoridades por Nivel de Gobierno y Condicion

In [ ]:
query_aut = """
SELECT 
    cg.tipo_gobierno AS Nivel_Gobierno,
    CASE WHEN fda.es_titular = 1 THEN 'Titular' ELSE 'Encargado (E)' END AS Condicion,
    COUNT(*) AS Cantidad
FROM fact_designacion_autoridades fda
JOIN dim_cargo cg ON fda.cargo_key = cg.cargo_key
GROUP BY cg.tipo_gobierno, fda.es_titular;
"""
df_aut = pd.read_sql_query(query_aut, conn)
display(df_aut)